# BERT Fine-tuning on IMDB Sentiment Analysis
This notebook fine-tunes BERT on the IMDB dataset, runs linear probing across layers, and visualises self-attention patterns for both the **base (pretrained)** and the **fine-tuned** model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# torch stuff
import torch

from torch.utils.data import DataLoader

# dataset
from datasets import load_dataset
dataset = load_dataset('imdb')

# model
from transformers import BertTokenizer, BertForSequenceClassification, AutoModelForSequenceClassification

# split training data into 90/10 train/validation split
split_dataset = dataset['train'].train_test_split(test_size=0.1, seed=42)
dataset_train = split_dataset['train']
dataset_val = split_dataset['test']
tokenizer = BertTokenizer.from_pretrained('bert-base-cased') # or "uncased"
encoded_dataset_train = dataset_train.map(lambda x: tokenizer(x['text'], truncation=True, max_length=256), batched=True) 
encoded_dataset_eval = dataset_val.map(lambda x: tokenizer(x['text'], truncation=True, max_length=256), batched=True) 
encoded_dataset_train.column_names
# Initialize a BERT model for binary classification
model_name = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
print(model.config)

In [ ]:
# [CLS] is a special token in BERT that is prepended to the beginning of each input sequence.
# It serves as a summary representation of the entire input, and its final hidden state
# (extracted as h[:, 0] in the code below) is commonly used for classification tasks.
#
# Linear probing: A technique to evaluate how much task-relevant information is already
# encoded in the pretrained model's representations. We freeze the BERT encoder and train
# a simple linear classifier (logistic regression) on top of the [CLS] embeddings from each layer.
# This shows where in the model the sentiment information becomes linearly separable,
# providing insights into BERT's inductive biases before fine-tuning.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader

PROBE_N_TRAIN = 2000
PROBE_N_EVAL = 1000
PROBE_BATCH = 16
PROBE_MAX_LEN = 256  # must match training max length so batches stack without a collator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


def _tokenize_fixed_length(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=PROBE_MAX_LEN,
    )


# Re-tokenize from raw text with fixed padding so every row is length PROBE_MAX_LEN
# (avoids variable-length rows from encoded_dataset_* / default DataLoader stack errors)
probe_train = (
    dataset_train.shuffle(seed=42)
    .select(range(min(PROBE_N_TRAIN, len(dataset_train))))
    .map(_tokenize_fixed_length, batched=True, remove_columns=["text"])
)
probe_eval = (
    dataset_val.shuffle(seed=42)
    .select(range(min(PROBE_N_EVAL, len(dataset_val))))
    .map(_tokenize_fixed_length, batched=True, remove_columns=["text"])
)
_probe_cols = ["input_ids", "attention_mask", "token_type_ids", "label"]
probe_train.set_format(type="torch", columns=_probe_cols)
probe_eval.set_format(type="torch", columns=_probe_cols)


@torch.no_grad()
def _cls_features_per_layer(bert, hf_dataset):
    loader = DataLoader(
        hf_dataset,
        batch_size=PROBE_BATCH,
        shuffle=False,
    )
    layer_chunks = None
    y_parts = []
    bert.eval()
    for batch in loader:
        labels = batch["label"] if "label" in batch else batch["labels"]
        inputs = {k: v.to(device) for k, v in batch.items() if k not in ("label", "labels")}
        out = bert(**inputs, output_hidden_states=True)
        stacked = torch.stack([h[:, 0].float().cpu() for h in out.hidden_states], dim=0)
        if layer_chunks is None:
            layer_chunks = [[] for _ in range(stacked.size(0))]
        for i in range(stacked.size(0)):
            layer_chunks[i].append(stacked[i])
        y_parts.append(labels.cpu())
    X_layers = [torch.cat(ch, dim=0).numpy() for ch in layer_chunks]
    y = torch.cat(y_parts, dim=0).numpy()
    return X_layers, y


Xtr, ytr = _cls_features_per_layer(model.bert, probe_train)
Xev, yev = _cls_features_per_layer(model.bert, probe_eval)

_layer_labels = ["embed"] + [f"L{i}" for i in range(model.config.num_hidden_layers)]
_acc, _f1 = [], []
for ell in range(len(Xtr)):
    clf = LogisticRegression(max_iter=500, n_jobs=-1)
    clf.fit(Xtr[ell], ytr)
    pred = clf.predict(Xev[ell])
    _acc.append(accuracy_score(yev, pred))
    _f1.append(f1_score(yev, pred, average="binary"))

for lab, a, f in zip(_layer_labels, _acc, _f1):
    print(f"{lab:5s}  acc={a:.3f}  F1={f:.3f}")

plt.figure(figsize=(8, 3))
plt.plot(range(len(_acc)), _acc, marker="o", label="accuracy")
plt.plot(range(len(_f1)), _f1, marker="s", label="F1 (binary)")
plt.xticks(range(len(_layer_labels)), _layer_labels, rotation=45, ha="right")
plt.ylabel("probe on val slice")
plt.title("Linear probe on [CLS] — frozen pretrained encoder (before finetuning)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Pretrained self-attention: one short sentence, one layer/head (qualitative only)
# SDPA / flash attention often returns an empty attentions tuple; eager materializes weights.
from transformers import AutoModel

_example = "Visually stunning but the story felt flat and forgettable."
_enc = tokenizer(_example, return_tensors="pt", truncation=True, max_length=64)
_enc = {k: v.to(device) for k, v in _enc.items()}

_viz_bert = AutoModel.from_pretrained(model_name, attn_implementation="eager")
_viz_bert.to(device).eval()
with torch.no_grad():
    _bout = _viz_bert(**_enc, output_attentions=True)

_attn = _bout.attentions
if _attn is None or len(_attn) == 0:
    raise RuntimeError("No attention weights returned; check transformers / attn_implementation.")

_layer_0based = 11
_head = 11
_layer_attn = _attn[_layer_0based]  # shape [batch, heads, seq, seq]
_A = _layer_attn[0, _head].float().cpu().numpy()
_toks = tokenizer.convert_ids_to_tokens(_enc["input_ids"][0])
plt.figure(figsize=(6, 5))
plt.imshow(_A, cmap="viridis")
plt.xticks(range(len(_toks)), _toks, rotation=90, fontsize=7)
plt.yticks(range(len(_toks)), _toks, fontsize=7)
plt.title(f"Pretrained attention — layer {_layer_0based + 1}, head {_head}")
plt.colorbar(fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
import wandb

wandb.login()
wandb_run = wandb.init(
    project="mini-project-aml-2026",
    entity="Mini-Project-AML-2026",
    name="bert-base-cased",
    config={
        "model_name": model_name,
        "num_labels": 2,
        "max_length": 256,
    }
)

In [ ]:
def try_gpu(i=0):
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

device = try_gpu()
print(f"Using device: {device}")

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
        num_train_epochs=3,              # Number of epochs
        per_device_train_batch_size=16,  # Batch size per GPU
        per_device_eval_batch_size=16,
        learning_rate=5e-5,              # Start with a small learning rate
        weight_decay=0.1,
        save_total_limit=2,              # Limit checkpoints to save space

        # Transformers-specific config
        logging_steps=100,
        load_best_model_at_end=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        bf16 = True,
        dataloader_pin_memory=True if torch.cuda.is_available() else False,
        output_dir=str("./results"),
        report_to = "wandb",
        run_name = "bert-base-cased",
        optim="adamw_torch",
)

In [ ]:
from transformers import Trainer
from evaluate import load

# Load a metric (F1-score in this case)
metric = load("f1")

# Define a custom compute_metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,                        # Pre-trained BERT model
    args=args,                          # Training arguments
    train_dataset=encoded_dataset_train,
    eval_dataset=encoded_dataset_eval,
    data_collator=data_collator,        # Efficient batching
    compute_metrics=compute_metrics     # Custom metric
)

# Start training
trainer.train()

# finish w&b logging
wandb.finish()

# Evaluate the model
results = trainer.evaluate()
print(results)

In [ ]:
sweep_config = {
    "method": "bayes",  # or "random" or "grid"
    "metric": {
        "name": "eval/f1",
        "goal": "maximize"
    },
    "parameters": {
        "model_name": {
            "values": ["bert-base-cased", "bert-base-uncased", "roberta-base"]
        },
        "learning_rate": {
            "values": [2e-5, 5e-5, 1e-4]
        },
        "num_train_epochs": {
            "values": [2, 3, 5]
        },
        "weight_decay": {
            "values": [0.0, 0.1]
        },
    }
}

sweep_id = wandb.sweep(sweep_config, project="mini-project-aml-2026")

def train():
    wandb.init()
    cfg = wandb.config  # access config params for this run

    # Load model using sweep param
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name, num_labels=2
    )

    args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=cfg.num_train_epochs,
        learning_rate=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        logging_steps=100,
        report_to="wandb",
        optim="adamw_torch",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=encoded_dataset["train"],
        eval_dataset=encoded_dataset["test"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    wandb.finish()

# Launch the sweep
wandb.agent(sweep_id, function=train, count=10)  # count --> max runs

---
## Attention Visualisation

Now that training is complete, we visualise self-attention patterns for:
1. **Base (pretrained) BERT** — `bert-base-cased` with no fine-tuning
2. **Fine-tuned BERT** — the model trained above on IMDB

We use two complementary views:
- **Heatmap grids** (matplotlib) — all 12 heads in a single layer at a glance
- **Interactive attention patterns** (circuitsvis) — scroll through heads interactively

This lets us compare how fine-tuning on sentiment shifts what each head attends to.

In [ ]:
# ── Install visualisation packages (Colab-safe) ──────────────────────────────
import os
try:
    import google.colab
    IN_COLAB = True
    print("Running in Colab — installing circuitsvis …")
    %pip install -q circuitsvis
    !curl -fsSL https://deb.nodesource.com/setup_16.x | sudo -E bash - && sudo apt-get install -y nodejs -qq
except ImportError:
    IN_COLAB = False
    print("Not in Colab — assuming circuitsvis is already installed.")

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import torch
import matplotlib.pyplot as plt
import circuitsvis as cv
from transformers import AutoModel, BertTokenizer

# Reuse the tokenizer already loaded above
# tokenizer = BertTokenizer.from_pretrained('bert-base-cased')  # already defined

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ── Helper: extract attention weights from a HuggingFace BERT-style model ────

@torch.no_grad()
def get_attention_weights(hf_model, tokenizer, text, device):
    """Return (attentions, str_tokens) for `text`.

    attentions: tuple of tensors, one per layer,
                each shape [batch=1, n_heads, seq_len, seq_len]
    str_tokens: list of token strings
    """
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=64)
    enc = {k: v.to(device) for k, v in enc.items()}
    hf_model.eval()
    out = hf_model(**enc, output_attentions=True)
    attentions = out.attentions  # tuple of [1, H, T, T]
    str_tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    return attentions, str_tokens


# ── Helper: heatmap grid — all heads for one layer ───────────────────────────

def plot_all_heads(attentions, str_tokens, layer_idx, title_prefix=""):
    """Plot a 3×4 grid of attention heatmaps for every head in `layer_idx`."""
    layer_attn = attentions[layer_idx][0].float().cpu()  # [H, T, T]
    n_heads = layer_attn.shape[0]
    n_cols = 4
    n_rows = (n_heads + n_cols - 1) // n_cols
    T = len(str_tokens)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.2, n_rows * 3))
    axes = axes.flatten()

    for h in range(n_heads):
        ax = axes[h]
        im = ax.imshow(layer_attn[h].numpy(), cmap="Blues", vmin=0)
        ax.set_title(f"Head {h}", fontsize=9)
        ax.set_xticks(range(T))
        ax.set_xticklabels(str_tokens, rotation=90, fontsize=6)
        ax.set_yticks(range(T))
        ax.set_yticklabels(str_tokens, fontsize=6)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for h in range(n_heads, len(axes)):
        axes[h].set_visible(False)

    fig.suptitle(f"{title_prefix}Layer {layer_idx + 1} — all {n_heads} heads", fontsize=12)
    plt.tight_layout()
    plt.show()


# ── Helper: circuitsvis interactive view ─────────────────────────────────────

def circuitsvis_attention(attentions, str_tokens, layer_idx, title_prefix=""):
    """Render an interactive circuitsvis attention pattern for `layer_idx`."""
    # cv expects attention shape [n_heads, seq, seq]
    layer_attn = attentions[layer_idx][0].float().cpu()  # [H, T, T]
    print(f"{title_prefix}Layer {layer_idx + 1} — interactive (scroll/click heads):")
    display(
        cv.attention.attention_patterns(
            tokens=str_tokens,
            attention=layer_attn,
        )
    )

In [ ]:
# ── Example sentence ─────────────────────────────────────────────────────────
# Feel free to swap this for any text you like.
EXAMPLE_TEXT = "Visually stunning but the story felt flat and forgettable."

# Which layer to focus on in the heatmap grid (0-indexed)
FOCUS_LAYER = 11  # last layer — most task-relevant in BERT

### 1 · Pretrained (base) BERT attention

We load `bert-base-cased` with `attn_implementation="eager"` so that HuggingFace returns
materialised attention weights (SDPA / FlashAttention skip this computation).

In [ ]:
# Load the raw encoder — no classification head needed for attention analysis
base_bert = AutoModel.from_pretrained(
    "bert-base-cased",
    attn_implementation="eager",  # ensures attention weights are materialised
)
base_bert.to(device).eval()

base_attentions, str_tokens = get_attention_weights(
    base_bert, tokenizer, EXAMPLE_TEXT, device
)
print(f"Tokens : {str_tokens}")
print(f"Layers : {len(base_attentions)}, Heads : {base_attentions[0].shape[1]}")

In [ ]:
# ── Heatmap grid: all 12 heads, last layer (pretrained) ──────────────────────
plot_all_heads(base_attentions, str_tokens, FOCUS_LAYER, title_prefix="[Pretrained] ")

In [ ]:
# ── Interactive circuitsvis view (pretrained) ─────────────────────────────────
circuitsvis_attention(base_attentions, str_tokens, FOCUS_LAYER, title_prefix="[Pretrained] ")

### 2 · Fine-tuned BERT attention

We extract the `bert` sub-module from the fine-tuned `BertForSequenceClassification` model
and wrap it with `attn_implementation="eager"` so attention weights are returned.

> **Note:** We reload the checkpoint via `AutoModel` pointing at the underlying encoder weights
> rather than the classification model, because the base `AutoModel` forward pass directly
> exposes `output_attentions`. An alternative is to use `model.bert` and call it directly.

In [ ]:
# The `model` variable still holds our fine-tuned BertForSequenceClassification.
# We extract the encoder sub-module and temporarily re-register it with eager attention.

# Option A (recommended): rebuild a standalone AutoModel from the fine-tuned encoder weights.
# Save the encoder state dict and reload it into a fresh eager model.
import tempfile, os

with tempfile.TemporaryDirectory() as tmp:
    # Save the bert encoder part to disk
    model.bert.save_pretrained(tmp)
    ft_bert = AutoModel.from_pretrained(
        tmp,
        attn_implementation="eager",
    )

ft_bert.to(device).eval()
print("Fine-tuned encoder loaded with eager attention.")

ft_attentions, str_tokens = get_attention_weights(
    ft_bert, tokenizer, EXAMPLE_TEXT, device
)
print(f"Tokens : {str_tokens}")
print(f"Layers : {len(ft_attentions)}, Heads : {ft_attentions[0].shape[1]}")

In [ ]:
# ── Heatmap grid: all 12 heads, last layer (fine-tuned) ──────────────────────
plot_all_heads(ft_attentions, str_tokens, FOCUS_LAYER, title_prefix="[Fine-tuned] ")

In [ ]:
# ── Interactive circuitsvis view (fine-tuned) ─────────────────────────────────
circuitsvis_attention(ft_attentions, str_tokens, FOCUS_LAYER, title_prefix="[Fine-tuned] ")

### 3 · Side-by-side comparison — single head

Pick any `(layer, head)` pair and view pretrained vs. fine-tuned attention side by side.

In [ ]:
COMPARE_LAYER = 11  # 0-indexed
COMPARE_HEAD  = 0   # 0-indexed

def side_by_side(base_attn, ft_attn, str_tokens, layer_idx, head_idx):
    A_base = base_attn[layer_idx][0, head_idx].float().cpu().numpy()
    A_ft   = ft_attn[layer_idx][0, head_idx].float().cpu().numpy()
    T = len(str_tokens)
    vmax = max(A_base.max(), A_ft.max())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    for ax, A, title in [
        (ax1, A_base, f"Pretrained — L{layer_idx+1} H{head_idx}"),
        (ax2, A_ft,   f"Fine-tuned  — L{layer_idx+1} H{head_idx}"),
    ]:
        im = ax.imshow(A, cmap="Blues", vmin=0, vmax=vmax)
        ax.set_title(title, fontsize=11)
        ax.set_xticks(range(T))
        ax.set_xticklabels(str_tokens, rotation=90, fontsize=7)
        ax.set_yticks(range(T))
        ax.set_yticklabels(str_tokens, fontsize=7)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.suptitle(f'Attention comparison: "{EXAMPLE_TEXT}"', fontsize=10)
    plt.tight_layout()
    plt.show()

side_by_side(base_attentions, ft_attentions, str_tokens, COMPARE_LAYER, COMPARE_HEAD)

### 4 · Layer-wise mean attention entropy

Entropy measures how *spread* or *focused* the attention distribution is.
A lower entropy → the head attends sharply to one or a few tokens.
Plotting mean entropy per layer lets us see at a glance whether fine-tuning
sharpens or diffuses attention.

In [ ]:
import numpy as np

def mean_entropy_per_layer(attentions):
    """Compute mean attention entropy per layer (averaged over heads and query positions)."""
    entropies = []
    for layer_attn in attentions:              # [1, H, T, T]
        A = layer_attn[0].float().cpu().numpy()  # [H, T, T]
        # entropy over key dimension, averaged over queries and heads
        # add tiny epsilon to avoid log(0)
        H = -np.sum(A * np.log(A + 1e-9), axis=-1)  # [H, T]
        entropies.append(H.mean())
    return entropies

base_ent = mean_entropy_per_layer(base_attentions)
ft_ent   = mean_entropy_per_layer(ft_attentions)
layers   = [f"L{i+1}" for i in range(len(base_ent))]

plt.figure(figsize=(9, 4))
plt.plot(layers, base_ent, marker="o", label="Pretrained", color="steelblue")
plt.plot(layers, ft_ent,   marker="s", label="Fine-tuned",  color="darkorange")
plt.ylabel("Mean attention entropy (nats)")
plt.title("Layer-wise mean attention entropy — pretrained vs. fine-tuned BERT")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()